In [ ]:
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, SGDRegressor, Ridge, Lasso
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error

import numpy as np
import pandas as pd
import joblib

import os

In [40]:
# Load the California housing dataset
fe_cal = fetch_california_housing(data_home='data')

print(fe_cal.DESCR)

print("特征值\n", fe_cal.data[:5])
print("目标值\n", fe_cal.target[:5])

.. _california_housing_dataset:

California Housing dataset
--------------------------

**Data Set Characteristics:**

:Number of Instances: 20640

:Number of Attributes: 8 numeric, predictive attributes and the target

:Attribute Information:
    - MedInc        median income in block group
    - HouseAge      median house age in block group
    - AveRooms      average number of rooms per household
    - AveBedrms     average number of bedrooms per household
    - Population    block group population
    - AveOccup      average number of household members
    - Latitude      block group latitude
    - Longitude     block group longitude

:Missing Attribute Values: None

This dataset was obtained from the StatLib repository.
https://www.dcc.fc.up.pt/~ltorgo/Regression/cal_housing.html

The target variable is the median house value for California districts,
expressed in hundreds of thousands of dollars ($100,000).

This dataset was derived from the 1990 U.S. census, using one row per ce

In [44]:
x_train, x_test, y_train, y_test = train_test_split(
    fe_cal.data, fe_cal.target, test_size=0.25, random_state=42)

# 标准化
std = StandardScaler()
x_train = std.fit_transform(x_train)
x_test = std.transform(x_test)

In [42]:
# 1. 线性回归模型
# 原理：线性回归模型试图找到一个线性函数，使得输入特征与目标变量之间的关系尽可能接近。
# 它通过最小化预测值与实际值之间的误差来拟合数据。
# 公式：y = β0 + β1*x1 + β2*x2 + ... + βn*xn + ε，其中β0是截距，β1, β2, ..., βn是特征的系数，ε是误差项。
# 正则方程：β = (X^T * X)^(-1) * X^T * y，其中X是特征矩阵，y是目标变量向量，β是回归系数向量。

model = LinearRegression()
model.fit(x_train, y_train)

print("回归系数\n", model.coef_)
print("截距\n", model.intercept_)

y_pred = model.predict(x_test)

if os.path.exists('model/linear_regression_model.pkl'):
    os.unlink('model/linear_regression_model.pkl')

joblib.dump(model, 'model/linear_regression_model.pkl')

print("均方误差\n", mean_squared_error(y_test, y_pred))

回归系数
 [ 0.85210815  0.12065533 -0.30210555  0.34860575 -0.00164465 -0.04116356
 -0.89314697 -0.86784046]
截距
 2.0703489205424743
均方误差
 0.5411287478470689


In [43]:
# 标准化 target
std_target = StandardScaler()
y_train = std_target.fit_transform(y_train.reshape(-1, 1)).flatten()

model.fit(x_train, y_train)
y_pred = model.predict(x_test)

# 将预测值反标准化
y_pred = std_target.inverse_transform(y_pred.reshape(-1, 1)).flatten()


print("回归系数\n", model.coef_)
print("截距\n", model.intercept_)
print("均方误差\n", mean_squared_error(y_test, y_pred))

回归系数
 [ 0.73767583  0.10445214 -0.26153484  0.30179038 -0.00142379 -0.03563558
 -0.77320342 -0.7512954 ]
截距
 -1.3935693996412475e-13
均方误差
 0.5411287478470691


In [47]:
# 梯度下降
# 原理：梯度下降是一种优化算法，用于最小化损失函数。它通过计算损失函数相对于模型参数的梯度，并沿着梯度的反方向更新参数来逐步逼近最优解。
# 公式：θ = θ - α * ∇J(θ)，其中θ是模型参数，α是学习率，∇J(θ)是损失函数J(θ)关于参数θ的梯度。

# 参数说明：
# max_iter：最大迭代次数，控制算法的收敛程度。
# tol：容忍度，控制算法的停止条件，当损失函数的变化小于tol时停止迭代。
# eta0：学习率，控制参数更新的步长。
# learning_rate：学习率的更新策略
# 常见的选项包括'constant'（固定学习率）、'optimal'（根据数据自动调整学习率）和'invscaling'（随着迭代次数增加逐渐减小学习率）。

sgd = SGDRegressor(eta0=0.01, max_iter=1000, tol=1e-3, random_state=42)
sgd.fit(x_train, y_train)

y_pred = sgd.predict(x_test)

print("回归系数\n", sgd.coef_)
print("截距\n", sgd.intercept_)
print("均方误差\n", mean_squared_error(y_test, y_pred))

回归系数
 [ 0.83264178  0.12498833 -0.25255367  0.4110813   0.00160827 -0.05168275
 -0.88794249 -0.87366852]
截距
 [2.05443863]
均方误差
 0.5854346671603641


In [ ]:
# L1 L2 正则化
# L1正则化（Lasso回归）通过在损失函数中添加特征系数的绝对值来惩罚模型复杂度。
# 这会导致一些特征的系数变为零，从而实现特征选择。
# J(θ) = MSE(θ) + α * ||θ||_1，其中MSE是均方误差，α是正则化强度，||θ||_1是参数θ的L1范数（即参数的绝对值之和）。
# L2正则化（Ridge回归）通过在损失函数中添加特征系数的平方来惩罚模型复杂度。
# 这会使得特征系数趋向于零，但不会完全变为零。
# J(θ) = MSE(θ) + α * ||θ||_2^2，其中MSE是均方误差，α是正则化强度，||θ||_2^2是参数θ的L2范数的平方（即参数的平方和）。